In [6]:
!pip install -q contractions

In [7]:
import pandas as pd
df = pd.read_csv("Suicide_Detection.csv")

In [8]:
df.head()

,Unnamed: 0,text,class
0,2,Ex Wife Threatening SuicideRecently I left my ...,suicide
1,3,Am I weird I don't get affected by compliments...,non-suicide
2,4,Finally 2020 is almost over... So I can never ...,non-suicide
3,8,i need helpjust help me im crying so hard,suicide
4,9,"I’m so lostHello, my name is Adam (16) and I’v...",suicide


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232074 entries, 0 to 232073
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  232074 non-null  int64 
 1   text        232074 non-null  object
 2   class       232074 non-null  object
dtypes: int64(1), object(2)
memory usage: 5.3+ MB


In [10]:
# preprocessing

duplicated_df = df[df.duplicated()]
duplicated_df

,Unnamed: 0,text,class


In [11]:
df.isna().sum()

Unnamed: 0    0
text          0
class         0
dtype: int64

In [12]:
df['class'].value_counts()

class
suicide        116037
non-suicide    116037
Name: count, dtype: int64

In [13]:
# preprocessing

import re
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup
import contractions

# Build stopword set — REMOVE negation words so they are preserved
stop = set(stopwords.words("english"))
negation_words = {
    "not", "no", "never", "neither", "nor", "nothing", "nowhere",
    "nobody", "none", "cannot", "without", "against", "hardly",
    "scarcely", "barely", "doesnt", "isnt", "wasnt", "shouldnt",
    "wouldnt", "couldnt", "wont", "cant", "dont", "didnt", "hadnt",
    "hasnt", "havent", "neednt", "mightnt", "mustnt",
}
stop = stop - negation_words


def expand_contractions(text):
    """Expand contractions so negations are preserved: don't → do not."""
    # Guard against short, empty, or non-string input that crashes contractions library
    if not isinstance(text, str) or len(text.strip()) < 2:
        return text if isinstance(text, str) else ""
    try:
        return contractions.fix(text)
    except (IndexError, Exception):
        # Fallback — contractions lib has a known IndexError bug on edge-case strings
        return text


def negate_sequence(text):
    negation_tokens = {
        "not", "no", "never", "nobody", "nothing", "nowhere",
        "neither", "nor", "cannot", "without", "hardly", "scarcely", "barely",
    }
    clause_breakers = {"but", "however", "although", "though", "yet"}

    tokens = text.split()
    result = []
    negating = False

    for token in tokens:
        clean_token = token.rstrip(".,!?;:")

        if clean_token in negation_tokens:
            negating = True
            result.append(token)
        elif clean_token in clause_breakers or token.endswith((".", "!", "?", ";")):
            negating = False
            result.append(token)
        elif negating and clean_token not in stop:
            result.append("NOT_" + clean_token)
        else:
            result.append(token)

    return " ".join(result)


def preprocess_text(text):
    # Handle NaN / None / non-string rows (common in real datasets)
    if not isinstance(text, str):
        return ""
    
    text = text.strip()
    if not text:
        return ""

    wl = WordNetLemmatizer()

    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # Expand contractions FIRST so "don't" → "do not" before any removal
    text = expand_contractions(text)

    # Remove emojis
    emoji_clean = re.compile(
        "["
        "\U0001f600-\U0001f64f"
        "\U0001f300-\U0001f5ff"
        "\U0001f680-\U0001f6ff"
        "\U0001f1e0-\U0001f1ff"
        "\U00002702-\U000027b0"
        "\U000024c2-\U0001f251"
        "]+",
        flags=re.UNICODE,
    )
    text = emoji_clean.sub(r"", text)

    # Clean up spacing and URLs
    text = re.sub(r"\.(?=\S)", ". ", text)
    text = re.sub(r"http\S+", "", text)

    # Lowercase, keep only letters and spaces
    text = re.sub(r"[^a-zA-Z\s]", "", text.lower())

    # Strip again after cleaning — may now be empty or too short
    text = text.strip()
    if not text:
        return ""

    # Apply negation tagging BEFORE stopword removal
    text = negate_sequence(text)

    # Remove stopwords and lemmatize
    tokens = []
    for word in text.split():
        if word.startswith("NOT_"):
            root = word[4:]
            if root.isalpha():
                tokens.append("NOT_" + wl.lemmatize(root))
        elif word not in stop and word.isalpha():
            tokens.append(wl.lemmatize(word))

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [14]:
'no' in stop

False

In [15]:
df['text'][0]

"Ex Wife Threatening SuicideRecently I left my wife for good because she has cheated on me twice and lied to me so much that I have decided to refuse to go back to her. As of a few days ago, she began threatening suicide. I have tirelessly spent these paat few days talking her out of it and she keeps hesitating because she wants to believe I'll come back. I know a lot of people will threaten this in order to get their way, but what happens if she really does? What do I do and how am I supposed to handle her death on my hands? I still love my wife but I cannot deal with getting cheated on again and constantly feeling insecure. I'm worried today may be the day she does it and I hope so much it doesn't happen."

In [16]:
preprocess_text(df['text'][0])

'ex wife threatening suiciderecently left wife good cheated twice lied much decided refuse go back day ago began threatening suicide tirelessly spent paat day talking keep hesitating want believe come back know lot people threaten order get way happens really supposed handle death hand still love wife cannot NOT_deal NOT_getting NOT_cheated NOT_constantly NOT_feeling NOT_insecure NOT_worried NOT_today NOT_may NOT_day NOT_hope NOT_much not NOT_happen'

In [ ]:
df['text'] = df['text'].apply(preprocess_text)

In [ ]:
df['text'][0]

In [ ]:
words_len = df['text'].str.split().map(lambda x: len(x))
df_temp = df.copy()
df_temp['words length'] = words_len
df_temp

In [ ]:
from collections import Counter

words = ' '.join(df['text']).split()
counter = Counter(words)

# remove stopwords
filtered = {word:count for word,count in counter.items() if word not in stop}

# most common
most_common = sorted(filtered.items(), key=lambda x: x[1], reverse=True)

# least common
least_common = sorted(filtered.items(), key=lambda x: x[1])

print("Most common:", most_common[:20])
print("Least common:", least_common[:20])

In [ ]:
# Text encoding

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

In [ ]:
x_data = df['text']

In [ ]:
x_data[:5]

In [ ]:
label_encode = LabelEncoder()
y_data = label_encode.fit_transform(df['class'])

In [ ]:
y_data[:5]

In [ ]:
print(label_encode.classes_)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x_data, y_data, test_size=0.2, random_state=42
)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),  
    min_df=2, 
)

In [ ]:
tfidf_vectorizer.fit(x_train)

In [ ]:
x_train_encoded = tfidf_vectorizer.transform(x_train)
x_test_encoded = tfidf_vectorizer.transform(x_test)

In [ ]:
x_train_encoded.shape

In [ ]:
# model training and evaluation
import time
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "LinearSVC": LinearSVC(),
    "MultinomialNB": MultinomialNB(),
    "SGD": SGDClassifier(),
    "XGBoost": XGBClassifier(eval_metric="logloss"),
}

results = []

for name, model in models.items():
    start = time.time()
    model.fit(x_train_encoded, y_train)
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(x_test_encoded)
    pred_time = time.time() - start

    results.append(
        [
            name,
            f1_score(y_test, y_pred, average="weighted"),
            precision_score(y_test, y_pred, average="weighted"),
            recall_score(y_test, y_pred, average="weighted"),
            train_time,
            pred_time,
        ]
    )

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "F1 Score",
        "Precision",
        "Recall",
        "Train Time",
        "Prediction Time",
    ],
)
print(results_df.sort_values("F1 Score", ascending=False).to_string(index=False))

LinearSVC achieved the best F1 score, but Logistic Regression was selected for deployment due to probability outputs required for user interpretability.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression()
lr_model.fit(x_train_encoded, y_train)
y_pred_lr = lr_model.predict(x_test_encoded)

In [ ]:
print("\n── Classification Report (SGD) ──")
print(classification_report(y_test, y_pred_lr))

In [ ]:
# test some sentences
text = "I can't take this anymore. I feel like ending my life"

processedText = preprocess_text(text)
X = tfidf_vectorizer.transform([processedText])
prediction = lr_model.predict(X)[0]

if prediction == 1:
    print("Potential suicide/ self-harm intention")
else:
    print("No violation")

Testing sentences:
* I can't take this anymore. I feel like ending my life
* I feel hopeless and want to end my life
* Sometimes I think about hurting myself
* Today was a beautiful day and I feel happy
